# 05 · EDA y prevalencia

**Pregunta que responde:** ¿cómo son realmente los datos de MIMIC-IV-ED sobre los que se
va a entrenar —qué falta, cómo se distribuyen los cinco parámetros medidos, cuánto dura
un episodio— y qué prevalencia tiene la etiqueta v2 en cada partición?

| | |
|---|---|
| **Issue** | `SCRUM-55` |
| **Notebook previo** | ninguno ejecutable todavía. Los pasos 01–04 del plan se implementaron como **módulos probados en `ml_engine/src/`** en vez de notebooks: `src/data/mimic_ed.py` (SCRUM-50), `src/data/etiquetas.py` (SCRUM-53) y `src/data/particiones.py` (SCRUM-54). Ver la nota al final. |
| **Entrada** | `ml_engine/data/processed/mimic-iv-ed-demo-2.2/observaciones_particionadas.parquet` y `episodios.parquet` |
| **Dataset** | MIMIC-IV-ED **demo v2.2** (subconjunto abierto, licencia ODbL). No es la base completa: `SCRUM-48` sigue esperando la credencial de PhysioNet. |
| **Salida** | `ml_engine/data/interim/eda_resumen.parquet` y las figuras en `ml_engine/data/interim/figuras/` |
| **Semilla** | `zaha-particion-v1` (la de `src/data/particiones.py`; este notebook no sortea nada por su cuenta) |

> ### ⚠️ Datos restringidos
> MIMIC-IV-ED viene bajo un DUA que **prohíbe redistribuirlo**, y este repositorio es
> público. **Este notebook no imprime ni una fila individual**: todo lo que sale de una
> celda es un agregado (conteos, proporciones, distribuciones). `nbstripout` limpia las
> salidas al commitear, pero eso es una red y no un permiso — las celdas de markdown el
> filtro no las toca.

> ### ⚠️ Lo que este notebook NO puede concluir
> Es descriptivo. Describe **datos**, no rendimiento de modelos, y por eso el hallazgo
> del horizonte de 24 h (§4) no lo invalida — al contrario, acá es donde ese hallazgo se
> muestra. Pero **ninguna tasa de alertas medida acá es la referencia** contra la cual se
> compara el modelo: esa referencia es del notebook `06_baseline_news2` y de `SCRUM-80`,
> y tiene que salir de la base completa.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd

from src.data import particiones
from src.viz import estilo

import matplotlib.pyplot as plt

estilo.aplicar()

# Reproducibilidad. Este notebook no sortea nada —la única aleatoriedad del proyecto es
# el reparto de particiones, que es determinístico por hash— pero la semilla se fija
# igual: una celda que un día use np.random no puede cambiar el resultado en silencio.
np.random.seed(20260921)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

DATOS = RAIZ / "data" / "processed" / "mimic-iv-ed-demo-2.2"
INTERIM = RAIZ / "data" / "interim"
FIGURAS = INTERIM / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

obs = pd.read_parquet(DATOS / "observaciones_particionadas.parquet")
epi = pd.read_parquet(DATOS / "episodios.parquet")

PARAMETROS = {
    "frecuencia_respiratoria": "Frec. respiratoria",
    "spo2": "SpO₂",
    "temperatura": "Temperatura",
    "presion_sistolica": "Presión sistólica",
    "frecuencia_cardiaca": "Frec. cardíaca",
}

print(f"observaciones: {len(obs)}   episodios: {len(epi)}   pacientes: {epi.paciente_id.nunique()}")

## 1 · La cohorte

**Qué espero encontrar antes de correr la celda:** un dataset chico y muy desbalanceado en
su estructura — pocos pacientes para muchos episodios, porque el demo de MIMIC se arma
tomando *todos* los episodios de un puñado de pacientes y no un paciente por episodio.
Espero también que la mayoría de los episodios termine en internación, porque ya lo
midió `etiquetas.py` (67,6 %).

Si eso se confirma, la consecuencia es que **el tamaño efectivo de la muestra no es 222
ni 1038, es 64**: la unidad estadística independiente es el paciente.

In [ ]:
tomas_por_episodio = obs.groupby("episodio_id").size()
episodios_por_paciente = epi.groupby("paciente_id").size()

cohorte = pd.Series({
    "pacientes": epi.paciente_id.nunique(),
    "episodios": len(epi),
    "observaciones": len(obs),
    "episodios por paciente (mediana)": episodios_por_paciente.median(),
    "episodios por paciente (máx)": episodios_por_paciente.max(),
    "tomas por episodio (mediana)": tomas_por_episodio.median(),
    "episodios con desenlace adverso": int(epi.desenlace_adverso.sum()),
    "% episodios con desenlace adverso": round(100 * epi.desenlace_adverso.mean(), 1),
    "pacientes con algún desenlace adverso": int(epi.groupby("paciente_id").desenlace_adverso.any().sum()),
})
print(cohorte.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
conteo = episodios_por_paciente.value_counts().sort_index()
barras = ax.bar(conteo.index.astype(str), conteo.values,
                color=estilo.TERRACOTA, **estilo.BORDE_OBLIGATORIO)
estilo.etiquetar_barras(ax, barras)
ax.set_xlabel("Episodios que tiene un mismo paciente")
ax.set_ylabel("Pacientes")
ax.set_title("La cola larga que obliga a partir por paciente", loc="left", color=estilo.TINTA)
ax.margins(y=0.18)
fig.savefig(FIGURAS / "05_episodios_por_paciente.png")
plt.show()

**Lo que se ve:** la mayoría de los pacientes tiene un episodio, pero la cola llega hasta
23. Ese paciente solo aporta el 10 % de los episodios del demo. Es la justificación
empírica de `SCRUM-54`: partir por fila o por episodio lo metería en entrenamiento y en
prueba a la vez.

## 2 · Qué falta

**Qué espero encontrar:** que la temperatura sea el parámetro más ausente, alrededor del
44 % — es lo que ya reportó la tubería. Espero que los otros cuatro estén casi completos.

Esto **no es ruido, es información sobre cómo se registra en guardia**: la temperatura
requiere un procedimiento aparte, mientras que frecuencia cardíaca y SpO₂ salen del mismo
monitor. Importa para el modelado porque una toma sin los cinco parámetros no es
puntuable por NEWS2, y ahí se decide cuántas filas sobreviven.

In [ ]:
faltantes = pd.DataFrame({
    "faltantes": [obs[c].isna().sum() for c in PARAMETROS],
    "% faltante": [round(100 * obs[c].isna().mean(), 1) for c in PARAMETROS],
}, index=list(PARAMETROS.values())).sort_values("% faltante", ascending=False)

print(faltantes.to_string())
print()
print(f"tomas puntuables (los 5 parámetros presentes): {int(obs.puntuable.sum())} de {len(obs)}"
      f"  ({100 * obs.puntuable.mean():.1f} %)")
print(f"valores anulados por estar fuera de rango fisiológico: {int(obs.valores_descartados.sum())}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
orden = faltantes.sort_values("% faltante")
barras = ax.barh(orden.index, orden["% faltante"],
                 color=estilo.TERRACOTA, **estilo.BORDE_OBLIGATORIO)
for barra, valor in zip(barras, orden["% faltante"]):
    ax.annotate(f"{valor:.1f} %", xy=(valor, barra.get_y() + barra.get_height() / 2),
                xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=9, color=estilo.TINTA)
ax.set_xlabel("% de tomas sin el parámetro")
ax.set_title("La temperatura es el cuello de botella de la puntuabilidad",
             loc="left", color=estilo.TINTA)
ax.margins(x=0.14)
ax.grid(axis="y", visible=False)
fig.savefig(FIGURAS / "05_faltantes_por_parametro.png")
plt.show()

## 3 · Cómo se distribuyen los cinco parámetros

**Qué espero encontrar:** distribuciones centradas en valores fisiológicos normales, con
colas hacia el lado patológico. Espero **poca masa en las zonas que NEWS2 puntúa alto**,
porque la mayoría de las tomas de guardia son de pacientes estables.

Si eso se confirma, explica por adelantado por qué la tasa de alertas de NEWS2 puede ser
alta aun cuando la mayoría de los pacientes está bien: basta con que la cola sea gruesa.

In [ ]:
fig, ejes = plt.subplots(1, 5, figsize=(14, 2.9))
for ax, (columna, titulo) in zip(ejes, PARAMETROS.items()):
    serie = obs[columna].dropna().astype(float)
    ax.hist(serie, bins=30, color=estilo.TERRACOTA, **estilo.BORDE_OBLIGATORIO)
    mediana = serie.median()
    ax.axvline(mediana, color=estilo.OLIVA, linewidth=2)
    ax.annotate(f"mediana {mediana:.0f}", xy=(mediana, ax.get_ylim()[1]),
                xytext=(0, -10), textcoords="offset points",
                ha="center", va="top", fontsize=8, color=estilo.OLIVA)
    ax.set_title(titulo, loc="left", fontsize=10, color=estilo.TINTA)
    ax.set_yticks([])
    ax.grid(axis="x", visible=False)
ejes[0].set_ylabel("Tomas")
fig.suptitle("Distribución de los cinco parámetros medidos", x=0.09, ha="left",
             fontsize=11, color=estilo.TINTA)
fig.tight_layout()
fig.savefig(FIGURAS / "05_distribucion_parametros.png")
plt.show()

In [ ]:
resumen_parametros = obs[list(PARAMETROS)].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T.round(1)
resumen_parametros.index = list(PARAMETROS.values())
print(resumen_parametros.to_string())

## 4 · La estadía, y por qué el horizonte de 24 h no discrimina en el demo

**Qué espero encontrar:** una estadía mediana de alrededor de 6 h y muy pocos episodios
que pasen las 24 h. `etiquetas.py` ya lo midió: mediana 5,8 h y sólo 6 episodios de 222
superan las 24 h.

**Por qué es el hallazgo más importante del notebook.** La etiqueta v2 define `y=1` si el
desenlace cae en `(t+1h, t+24h]`. Si prácticamente **toda** la estadía entra en esa
ventana, entonces la pregunta *"¿el desenlace ocurre dentro de 24 h?"* tiene la misma
respuesta que *"¿el episodio terminó en internación?"* — y la etiqueta, que pretende ser
a nivel de toma, colapsa en una etiqueta a nivel de episodio.

Esto **no invalida la definición de la etiqueta**, que es la correcta. Invalida al demo
como banco de pruebas.

In [ ]:
horas = epi["horas_en_guardia"].dropna()
print(f"estadía mediana        : {horas.median():.1f} h")
print(f"percentil 90           : {horas.quantile(0.90):.1f} h")
print(f"episodios sobre 24 h   : {int((horas > 24).sum())} de {len(horas)}  ({100*(horas>24).mean():.1f} %)")
print()
etiquetables = obs["y"].notna()
print(f"tomas etiquetables     : {int(etiquetables.sum())} de {len(obs)}")
print(f"  descartadas, ventana ciega  : {int(obs.en_ventana_ciega.sum())}")
print(f"  descartadas, dato posterior : {int(obs.posterior_al_evento.sum())}")
print()
fuera = obs.loc[obs.horas_al_evento.notna(), "horas_al_evento"] > 24
print(f"tomas con evento FUERA del horizonte de 24 h: {int(fuera.sum())} de {int(fuera.size)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(horas.clip(0, 48), bins=48, color=estilo.TERRACOTA, **estilo.BORDE_OBLIGATORIO)
ax.axvline(24, color=estilo.OLIVA, linewidth=2)
ax.annotate("horizonte de la etiqueta: 24 h\n← casi toda la masa queda de este lado",
            xy=(24, ax.get_ylim()[1] * 0.92), xytext=(-8, 0),
            textcoords="offset points", ha="right", va="top",
            fontsize=9, color=estilo.OLIVA, linespacing=1.5)
ax.axvline(horas.median(), color=estilo.GRIS, linewidth=1.5, linestyle=(0, (4, 3)))
# La etiqueta de la mediana cae justo sobre las barras mas altas: se la sube por encima
# del pico y se la ancla con una guia, en vez de dejarla pisando los datos.
ax.annotate(f"mediana {horas.median():.1f} h",
            xy=(horas.median(), ax.get_ylim()[1] * 0.58),
            xytext=(30, 2), textcoords="offset points",
            fontsize=9, color=estilo.GRIS,
            arrowprops=dict(arrowstyle="-", color=estilo.GRIS, linewidth=1,
                            shrinkA=0, shrinkB=2))

# El ultimo bin acumula TODO lo que pasa de 48 h por el clip. Sin decirlo, esa barra se
# lee como "3 episodios de 48 h" y en realidad es "3 episodios de 48 h o mas".
sobre = int((horas > 48).sum())
if sobre:
    ax.annotate(f"{sobre} episodios superan" + chr(10) + f"las 48 h y se apilan aqui", xy=(48, sobre),
                xytext=(-6, 26), textcoords="offset points",
                ha="right", fontsize=8.5, color=estilo.GRIS, linespacing=1.4,
                arrowprops=dict(arrowstyle="-", color=estilo.GRIS, linewidth=1,
                                shrinkA=0, shrinkB=3))
ax.set_xlabel("Horas en guardia (el ultimo bin acumula todo lo que pasa de 48 h)")
ax.set_ylabel("Episodios")
ax.set_title("El horizonte de 24 h abarca casi todos los episodios del demo",
             loc="left", color=estilo.TINTA)
fig.savefig(FIGURAS / "05_estadia_vs_horizonte.png")
plt.show()

**Lo que se ve:** la línea de las 24 h deja casi toda la distribución a su izquierda. En
la base completa —y sobre todo en datos de **sala general**, donde la estadía se mide en
días— el horizonte sí separa. Acá no.

**Consecuencia operativa:** ninguna métrica de modelado calculada sobre el demo con esta
etiqueta es interpretable. El código está listo; el dataset no alcanza. Va declarado
junto al caveat del ADR-006.

## 5 · Prevalencia, global y por partición

**Qué espero encontrar:** una prevalencia altísima —cerca del 67 %— que **no es una
prevalencia clínica**. Es el reflejo directo de §4: como el horizonte abarca todo, `y=1`
equivale a "el episodio terminó en internación".

Para contraste: la prevalencia real del problema que este proyecto quiere resolver está
entre 0,22 % y 5 %. Un 67 % es la señal de que el demo no representa el problema.

Espero además que las tres particiones tengan prevalencias parecidas pero no idénticas,
porque con 4 pacientes negativos la estratificación no tiene con qué equilibrar.

In [ ]:
resumenes = particiones.resumir(obs)
tabla = pd.DataFrame([{
    "partición": r.nombre,
    "pacientes": r.pacientes,
    "episodios": r.episodios,
    "observaciones": r.observaciones,
    "etiquetables": r.etiquetables,
    "y=1": r.positivas,
    "prevalencia %": round(100 * r.prevalencia, 1),
} for r in resumenes]).set_index("partición")

print(tabla.to_string())
print()
glob = obs.loc[obs.y.notna(), "y"]
print(f"prevalencia global: {100 * glob.mean():.1f} %  ({int(glob.sum())} de {len(glob)})")
print()
print("Pacientes por estrato:")
print(epi.groupby("paciente_id").desenlace_adverso.any().value_counts().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 3.6))
barras = ax.bar(tabla.index, tabla["prevalencia %"],
                color=estilo.TERRACOTA, **estilo.BORDE_OBLIGATORIO)
estilo.etiquetar_barras(ax, barras, "{:.1f} %")

# La referencia del 5 % y el tamaño de cada partición van FUERA de las barras: texto
# gris u oliva encima del terracota no se lee, y además el texto nunca lleva el color
# de la serie. El n de cada partición baja al eje, que es donde el lector ya mira.
ax.axhline(5, color=estilo.OLIVA, linewidth=2)
ax.set_xlim(-0.6, len(tabla) - 0.1 + 0.9)
ax.annotate("5 % — techo de la\nprevalencia real",
            xy=(len(tabla) - 0.45, 5), ha="left", va="center",
            fontsize=8.5, color=estilo.OLIVA, linespacing=1.4)

ax.set_xticks(range(len(tabla)))
ax.set_xticklabels([f"{n}\n{p} pac · {e} tomas"
                    for n, p, e in zip(tabla.index, tabla["pacientes"], tabla["etiquetables"])])

ax.set_ylabel("Prevalencia de y=1 (%)")
ax.set_title("La prevalencia del demo está un orden de magnitud por encima de la real",
             loc="left", color=estilo.TINTA)
ax.margins(y=0.20)
ax.grid(axis="x", visible=False)
fig.savefig(FIGURAS / "05_prevalencia_por_particion.png")
plt.show()

## 6 · Distribución del NEWS2 en esta cohorte

**Qué espero encontrar:** la mayoría de las tomas puntuables en riesgo Bajo, con una cola
hacia Medio y Alto.

**Nota sobre el color.** El nivel de riesgo es una variable **ordinal**, así que va en una
rampa secuencial de un solo tono y no en los cuatro colores de la interfaz clínica. Esos
cuatro fallan el control de daltonismo como paleta de gráfico —Medio ↔ Medio Bajo dan
ΔE 3,0 en deuteranopia— y en la interfaz sólo son legales porque van acompañados de texto
y forma. El detalle está en `src/viz/estilo.py`.

**Lo que esta sección NO hace:** no fija la tasa de alertas de referencia. Contar qué
fracción de tomas supera NEWS2 ≥ 5 acá sería tentador y sería un error: la referencia
tiene que medirse sobre la cohorte completa y con la definición de tasa por
paciente-día, y eso es el notebook `06_baseline_news2` y `SCRUM-80`.

In [ ]:
puntuables = obs[obs.puntuable]
NIVELES = ["Bajo", "Medio Bajo", "Medio", "Alto"]
conteo = puntuables.news2_riesgo.value_counts().reindex(NIVELES, fill_value=0)

fig, ax = plt.subplots(figsize=(7.5, 3.4))
barras = ax.bar(conteo.index, conteo.values,
                color=estilo.rampa(4), **estilo.BORDE_OBLIGATORIO)
# Conteo y porcentaje en UNA sola etiqueta. Como dos anotaciones separadas se
# superponen en las barras cortas (Medio y Alto son 19 y 10 sobre un eje que llega a
# 491) y el resultado es ilegible.
for barra, valor in zip(barras, conteo.values):
    ax.annotate(f"{valor}  ({100 * valor / conteo.sum():.0f} %)",
                xy=(barra.get_x() + barra.get_width() / 2, barra.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=9, color=estilo.TINTA)
ax.set_ylabel("Tomas puntuables")
ax.set_title("Nivel de riesgo NEWS2 — escala ordinal, rampa de un solo tono",
             loc="left", color=estilo.TINTA)
ax.margins(y=0.20)
ax.grid(axis="x", visible=False)
fig.savefig(FIGURAS / "05_niveles_news2.png")
plt.show()

print(f"tomas puntuables: {len(puntuables)}")
print(f"score NEWS2 — mediana {puntuables.news2_score.median():.0f}, "
      f"p90 {puntuables.news2_score.quantile(0.9):.0f}, máx {puntuables.news2_score.max():.0f}")
print(f"tomas con al menos un parámetro imputado: {int(puntuables.news2_imputado.sum())}"
      f"  ({100 * puntuables.news2_imputado.mean():.0f} %)")

**Ojo con la imputación: alcanza al 100 % de las tomas.** MIMIC-IV-ED no registra
consciencia ni oxígeno suplementario en *ninguna* fila, así que los dos se imputan
siempre al valor de **menor** riesgo (ADR-007). No es que falten a veces: faltan
siempre. El NEWS2 calculado acá es por lo tanto un **piso**, nunca el score real — un
paciente confundido o con oxígeno puntuaría más, y el sesgo va siempre en la misma
dirección, hacia abajo.

Esto tiene una consecuencia directa sobre la comparativa de `SCRUM-58`: si la tasa de
alertas de referencia de NEWS2 se mide sobre esta cohorte, está **subestimada**. Y
subestimar la referencia hace que el modelo propio parezca peor de lo que es, no mejor
— es decir que el sesgo juega en contra de la hipótesis del proyecto, no a favor. Eso es
defendible; lo contrario no lo sería.

In [ ]:
# Salida para el notebook siguiente. Agregados únicamente: ni una fila individual sale
# de data/, que está entera en .gitignore.
salida = pd.concat([
    faltantes.reset_index().rename(columns={"index": "clave"}).assign(bloque="faltantes"),
    tabla.reset_index().rename(columns={"partición": "clave"}).assign(bloque="particiones"),
], ignore_index=True)

INTERIM.mkdir(parents=True, exist_ok=True)
salida.to_parquet(INTERIM / "eda_resumen.parquet", index=False)
print(f"guardado: {INTERIM / 'eda_resumen.parquet'}  ({len(salida)} filas de agregados)")
print(f"figuras : {FIGURAS}  ({len(list(FIGURAS.glob('05_*.png')))} archivos)")

## Conclusiones

**Qué se concluyó**

1. **El tamaño efectivo de la muestra es 64, no 1038.** La unidad independiente es el
   paciente; 222 episodios salen de 64 pacientes y uno solo aporta 23. Confirma
   empíricamente la decisión de `SCRUM-54`.
2. **La temperatura falta en ~44 % de las tomas** y es lo que decide la puntuabilidad.
   Los otros cuatro parámetros están casi completos. Es información sobre el registro en
   guardia, no ruido.
3. **El horizonte de 24 h no discrimina en el demo.** La estadía mediana es de ~6 h y casi
   ningún episodio pasa las 24 h, así que la etiqueta colapsa a "terminó en internación".
   **Es el hallazgo que condiciona todo el Sprint de modelado.**
4. **La prevalencia del demo (~67 %) está un orden de magnitud por encima de la real**
   (0,22 %–5 %). No es una prevalencia clínica.
5. **Con 4 pacientes negativos de 64**, ninguna partición de prueba del demo puede dar una
   prevalencia interpretable, por más correcta que sea la estratificación.
6. **El NEWS2 calculado acá es un piso**, porque consciencia y oxígeno suplementario se
   imputan al valor de menor riesgo (ADR-007).

**Qué se guardó**

- `data/interim/eda_resumen.parquet` — los agregados de faltantes y de particiones.
- `data/interim/figuras/05_*.png` — seis figuras, en el estilo único de `src/viz/estilo.py`.

**Qué queda abierto**

- **Todo lo métrico depende de la base completa** (`SCRUM-48`, credencial de PhysioNet).
  Este notebook se vuelve a correr apuntando a otra carpeta y no cambia una línea.
- El siguiente es **`06_baseline_news2`**, que reproduce NEWS2 sobre esta cohorte y mide
  su tasa de alertas de referencia. **Ese notebook no tiene issue propio en el backlog**
  y es imprescindible: comparar el modelo contra el 37,6 por 100 pacientes-día de la
  literatura —medido sobre otra población— es indefendible. Candidato natural a ser
  `SCRUM-80`, que está pendiente de reescritura.

**Nota sobre la numeración**

El plan (`Plan/05_PREPARACION.md`) preveía los notebooks `01`–`04` antes de éste. Ese
trabajo se hizo, pero como **módulos con pruebas en `ml_engine/src/`** en lugar de
notebooks: `mimic_ed.py`, `etiquetas.py` y `particiones.py`, con 66 pruebas entre los
cuatro archivos de test. Es consistente con la regla del proyecto —*el notebook explora,
`src/` ejecuta*— y es mejor que un notebook para código que la API va a usar en
producción. Los números `01`–`04` quedan libres a propósito: si más adelante hace falta
la narrativa exploratoria de esos pasos, entra en su lugar.